In [1]:
%pip install -U google-genai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import json

import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError("GEMINI_API_KEY não configurada")

client = genai.Client(api_key=api_key)

## Parte A - Tratamento e regras (pandas)

In [4]:
with open('../dados/dados_nivel_1.json', 'r', encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])

print(taxa_cambio)
df.shape

5.4


(20, 9)

In [5]:
df.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [6]:
df.head(10)

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB


In [8]:
df.describe()

,valor
count,20.000000
mean,11495.000000
std,7983.369227
min,1400.000000
25%,4175.000000
50%,10400.000000
75%,17225.000000
max,27000.000000


In [9]:
df.isnull().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [10]:
df[df['data'].isnull()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [11]:
df[df['cliente_id'].eq('CLI-A-5')]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
14,OP-0014,CLI-A-5,2026-03-07,2900,BRL,pix,transferencia_recebida,Delta Transportes,
15,OP-0015,CLI-A-5,2026-03-16,7000,BRL,ted,transferencia_recebida,Epsilon Consultoria,
16,OP-0016,CLI-A-5,2026-03-26,2700,BRL,boleto,pagamento,Beta Servicos ME,
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [12]:
df[df.duplicated(keep=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [13]:
df = df.drop_duplicates().reset_index(drop=True)

In [14]:
df[df.duplicated(keep=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao


In [15]:
df["moeda"].value_counts()

moeda
BRL    18
USD     1
Name: count, dtype: int64

In [16]:
df[df["moeda"].str.strip().str.upper() != "BRL"]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


In [17]:
df["valor_brl"] = df["valor"] * df["moeda"].str.strip().str.upper().map({"BRL": 1.0, "USD": taxa_cambio})
df[df["moeda"].str.strip().str.upper() != "BRL"]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,64800.0


In [18]:
# volume total transacionado por cliente
volume_por_cliente = (
    df.groupby("cliente_id", as_index=False)["valor_brl"]
      .sum()
      .rename(columns={"valor_brl": "volume_total_brl"})
)
volume_por_cliente

,cliente_id,volume_total_brl
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
3,CLI-A-4,79500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [19]:
operacoes_por_canal = (
    df.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
      .sort_values("quantidade_operacoes", ascending=False)
)
operacoes_por_canal

,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


### Regra 1 - Fracionamento:
Cliente que, em uma mesma data:
* realizou 3 ou mais operações cuja soma ultrapassa BRL 50.000,00;
* nenhuma operação isolada atinge BRL 20.000,00

In [20]:
df.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,25900.0


In [21]:
chave = ["cliente_id", "data"]

qtd = df.groupby(chave)["valor_brl"].transform("size")
soma = df.groupby(chave)["valor_brl"].transform("sum")
maxi = df.groupby(chave)["valor_brl"].transform("max")

df["flag_frac"] = (qtd >= 3) & (soma > 50000) & (maxi < 20000)


In [22]:
df[df["flag_frac"].eq(True)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,flag_frac
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0,True


In [23]:
df.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,flag_frac
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0,True
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,3300.0,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,25900.0,False


### Regra 2 - Valor atípico. 
Sinalize a operação cujo valor em BRL seja superior a 5× a mediana dos valores daquele mesmo cliente. Aplique apenas a clientes com 4 ou mais operações.

In [24]:
qtd = df.groupby("cliente_id")["valor_brl"].transform("size")
mediana = df.groupby("cliente_id")["valor_brl"].transform("median")
df["flag_atipico"] = (qtd >= 4) & (df["valor_brl"] > 3 * mediana)

In [25]:
df.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,flag_frac,flag_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0,True,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,3300.0,False,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,25900.0,False,False


In [26]:
df[df["flag_atipico"].eq(True)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,flag_frac,flag_atipico
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,64800.0,False,True


In [27]:
teste = pd.DataFrame([
    # cliente, dia, valor, esperado
    ("A", "2026-01-01", 19999, True), # positivo: 3 operações, soma 50001, max 19999
    ("A", "2026-01-01", 19999, True),
    ("A", "2026-01-01", 10003, True),

    ("B", "2026-01-01", 19999, False),
    ("B", "2026-01-01", 19999, False),
    ("B", "2026-01-01", 10002, False),
    
    ("C", "2026-01-01", 20000, False),
    ("C", "2026-01-01", 19999, False),
    ("C", "2026-01-01", 10002, False),
    
    ("C", "2026-01-01", 20000, False),
    ("C", "2026-01-01", 19999, False),
    ("C", "2026-01-02", 10003, False),
    
], columns=["cliente", "dia", "valor", "esperado"])

g = teste.groupby(["cliente", "dia"])["valor"]
teste["flag"] = (
    (g.transform("size") >= 3)
    & (g.transform("sum") > 50000)
    & (g.transform("max") < 20000)
)

assert (teste["flag"] == teste["esperado"]).all(), "Regra divergiu do esperado"
print(teste)

   cliente         dia  valor  esperado   flag
0        A  2026-01-01  19999      True   True
1        A  2026-01-01  19999      True   True
2        A  2026-01-01  10003      True   True
3        B  2026-01-01  19999     False  False
4        B  2026-01-01  19999     False  False
5        B  2026-01-01  10002     False  False
6        C  2026-01-01  20000     False  False
7        C  2026-01-01  19999     False  False
8        C  2026-01-01  10002     False  False
9        C  2026-01-01  20000     False  False
10       C  2026-01-01  19999     False  False
11       C  2026-01-02  10003     False  False


Se nenhuma operação chega a 20.000 e a soma passa de 50.000, então são necessariamente 3 ou mais operações (2 operações dariam menos de 40.000). Por isso você não consegue montar um quase-caso que falhe só em qtd >= 3. o enunciado a pede e ela documenta a intenção, mas o relatório pode registrar essa observação como análise da regra

In [28]:
df[df["flag_frac"].eq(True) | df["flag_atipico"].eq(True)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,flag_frac,flag_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0,True,False
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,64800.0,False,True


## Parte B - Análise com LLM

In [30]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explique brevemente o que é uma API REST."
)

print(response.text)

Uma **API REST** é uma forma padronizada de permitir que dois sistemas diferentes conversem e troquem dados pela internet, utilizando as mesmas regras e protocolos da própria web (o protocolo **HTTP**).

Pense nela como um **garçom em um restaurante**:
* Você (o **Cliente**) faz um pedido através do cardápio.
* O garçom (a **API**) leva seu pedido até a cozinha (o **Servidor**).
* A cozinha prepara o prato e o garçom traz a resposta de volta para você.

### Principais características:

1. **Usa os comandos padrões da Web (Verbos HTTP):**
   * **GET:** Para buscar uma informação (ex: ver a lista de produtos).
   * **POST:** Para criar algo novo (ex: cadastrar um novo usuário).
   * **PUT/PATCH:** Para alterar um dado existente (ex: atualizar seu endereço).
   * **DELETE:** Para apagar um dado (ex: excluir uma conta).

2. **Formato JSON:** Quase sempre responde usando **JSON**, um formato de texto simples e leve, fácil de ser lido tanto por humanos quanto por qualquer linguagem de progra